google collab depedencies

In [4]:
# !pip -q install bertopic
# !pip -q install sastrawi
# !pip -q install gensim

In [5]:
# !git clone -q -b gavriel-thesis https://github.com/ranslemus/topic_modeling_KBMI4.git
# %cd topic_modeling_KBMI4

In [6]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import plotly.express as px

from transformers import AutoTokenizer, AutoModel
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from tqdm.auto import tqdm
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from hdbscan.validity import validity_index

# for linux
# from cuml.manifold import UMAP
# from cuml.cluster import HDBSCAN

# for windows
# import umap as UMAP
# import hdbscan as HDBSCAN

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device :", device)

if device.type == "cuda":
    print("GPU :", torch.cuda.get_device_name(0))

Device : cuda
GPU : NVIDIA GeForce GTX 1650 Ti


In [8]:
df = pd.read_csv("data/preprocessed_data_downsampled.csv")

df.head()

,reviewId,bank,score,year,text
0,96c7507f-8be5-40a9-8b3a-009ac2036630,WONDR_BNI_REVIEWS,2,2024,kebanyakan maintenance jadi enggak efektif pak...
1,e0c5d420-5bb9-4408-854e-4f6744dd432b,WONDR_BNI_REVIEWS,1,2024,sampai saat ini tidak bisa diakses sudah dinon...
2,f977cd45-7e77-4735-a7de-7e4e8b827db3,LIVIN_MANDIRI_REVIEWS,1,2023,enggak bisa dibuka padahal usah coba berbagai ...
3,629f06db-dc19-4a6b-a526-c5fa09933ed2,LIVIN_MANDIRI_REVIEWS,2,2025,kenapa di login tidak bisa ya malah muncul tul...
4,d1116af0-d952-49b8-9ff9-692f6093f599,BRIMO_REVIEWS,1,2023,buruk data sudah benar malah enggak bisa dikon...


In [9]:
# df["word_count"] = df["text"].astype(str).str.split().apply(len)
# df = df[df["word_count"] >= 5].reset_index(drop=True)
# print(f"Total documents setelah filter: {len(df):,}")

In [10]:
documents = df["text"].astype(str).tolist()

print(f"Total documents : {len(documents):,}")

Total documents : 155,943


# IndoBERT

In [11]:
MODEL_NAME = "indobenchmark/indobert-base-p1"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModel.from_pretrained(MODEL_NAME)

model.to(device)

model.eval()

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(50000, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


In [12]:
def mean_pooling(model_output, attention_mask):

    token_embeddings = model_output.last_hidden_state

    input_mask_expanded = (
        attention_mask
        .unsqueeze(-1)
        .expand(token_embeddings.size())
        .float()
    )

    return torch.sum(
        token_embeddings * input_mask_expanded,
        dim=1
    ) / torch.clamp(
        input_mask_expanded.sum(dim=1),
        min=1e-9
    )

In [13]:
def encode_documents(
    documents,
    batch_size=32,
    max_length=128
):

    embeddings = []

    with torch.no_grad():

        for i in tqdm(
            range(0, len(documents), batch_size)
        ):

            batch = documents[
                i:i+batch_size
            ]

            encoded_input = tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt"
            )

            encoded_input = {
                k: v.to(device)
                for k, v in encoded_input.items()
            }

            model_output = model(**encoded_input)

            sentence_embeddings = mean_pooling(
                model_output,
                encoded_input["attention_mask"]
            )

            sentence_embeddings = (
                sentence_embeddings
                .cpu()
                .numpy()
            )

            embeddings.append(sentence_embeddings)

    return np.vstack(embeddings)

In [14]:
embeddings = encode_documents(
    documents,
    batch_size=32,
    max_length=128
)

  0%|          | 0/4874 [00:00<?, ?it/s]

100%|██████████| 4874/4874 [22:51<00:00,  3.55it/s]


In [15]:
print(embeddings.shape)

(155943, 768)


In [16]:
embeddings[0]

array([ 1.99362922e+00,  8.71141911e-01,  2.32575998e-01,  4.88159597e-01,
       -1.53943285e-01,  6.54066205e-02, -7.94848919e-01,  6.44386113e-02,
        5.35463095e-01, -1.38590842e-01, -8.13299343e-02, -1.04618859e+00,
       -9.04206872e-01,  8.62348080e-01,  2.20929645e-02, -3.00666213e-01,
       -5.28009176e-01, -1.01320960e-01, -2.91481726e-02,  7.27127373e-01,
        3.72639477e-01, -3.66047055e-01, -6.55409694e-02, -1.06161678e+00,
       -7.23849654e-01, -2.64188051e-01, -7.38128871e-02,  3.38941544e-01,
       -4.32385951e-01, -4.99101490e-01,  7.13105559e-01,  4.50596333e-01,
       -7.68126175e-03,  3.48132193e-01, -1.58587205e+00,  1.17839050e+00,
       -4.46127534e-01,  1.15841866e+00, -1.05954146e+00, -1.42351165e-01,
       -1.08025980e+00,  3.64045322e-01, -1.19255567e+00, -4.37219262e-01,
       -3.45705748e-01,  7.09880769e-01,  3.18425834e-01,  1.71062791e+00,
        4.44135144e-02,  1.10975653e-01, -1.06760168e+00, -7.84462392e-01,
        2.28673518e-01,  

In [17]:
norms = np.linalg.norm(embeddings, axis=1)

print("Minimum Norm :", norms.min())
print("Maximum Norm :", norms.max())
print("Average Norm :", norms.mean())
print("Std Norm :", norms.std())

Minimum Norm : 12.87398
Maximum Norm : 26.135761
Average Norm : 18.315
Std Norm : 2.1675787


In [18]:
print("NaN :", np.isnan(embeddings).sum())
print("Inf :", np.isinf(embeddings).sum())

NaN : 0
Inf : 0


In [19]:
np.save(
    "embeddings/indobert_embeddings_downsampled_full.npy",
    embeddings
)

# BERTopic

In [ ]:
embeddings = np.load("indobert_embeddings_downsampled.npy")

print("Embedding Shape :", embeddings.shape)

Embedding Shape : (168653, 768)


In [124]:
sastrawi_stopwords = StopWordRemoverFactory().get_stop_words()

# extra_particles = ["banget", "terus", "padahal", "sih", "aja", "saja", "dong", "deh", "ya", "kok", "biar", "gitu", "nih", "loh", "mau", "sudah", "belum"]
# sastrawi_stopwords_extended = sastrawi_stopwords + extra_particles

vectorizer_model = CountVectorizer(
  ngram_range=(1,2),
  stop_words=sastrawi_stopwords,
  token_pattern=r"(?u)\b[^\d\W]+\b",
  min_df=2,
  )

baseline UMAP for testing purpose

In [125]:
umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    metric="cosine",
    min_dist=0.0,
    random_state=42
)

baseline HDBSCAN

In [126]:
hdbscan_model = HDBSCAN(
    min_cluster_size=100,
    min_samples=10,
    metric="euclidean",
    cluster_selection_method="leaf",
    prediction_data=True
)

In [127]:
topic_model = BERTopic(
    embedding_model=None,
    calculate_probabilities=False,
    vectorizer_model=vectorizer_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    verbose=True
)

In [128]:
topics, probabilities = topic_model.fit_transform(
    documents,
    embeddings
)

2026-08-10 03:15:44,991 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-08-10 03:15:59,032 - BERTopic - Dimensionality - Completed ✓
2026-08-10 03:15:59,044 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-08-10 03:16:02,964 - BERTopic - Cluster - Completed ✓
2026-08-10 03:16:03,023 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-08-10 03:16:14,540 - BERTopic - Representation - Completed ✓


# Evaluation

Basic Statistics

In [129]:
topic_info = topic_model.get_topic_info()

topic_info.head(10)

,Topic,Count,Name,Representation,Representative_Docs
0,-1,142860,-1_enggak_aplikasi_mau_nya,"[enggak, aplikasi, mau, nya, terus, padahal, m...",[sudah hampir sebulanan enggak bisa buka aplik...
1,0,1902,0_username_benar_password_salah,"[username, benar, password, salah, pasword, sa...",[kenapa enggak bisa login padahal username sam...
2,1,1800,1_buka_dibuka_enggak buka_kok,"[buka, dibuka, enggak buka, kok, enggak dibuka...",[kenapa enggak bisa di buka saat ini aplikasi ...
3,2,1564,2_perbaiki_diperbaiki_mohon_segera,"[perbaiki, diperbaiki, mohon, segera, sering, ...",[bug force close terus ni mohon segera diperba...
4,3,1526,3_uninstall_tetap_install_instal,"[uninstall, tetap, install, instal, ulang, uni...",[bagaimana ini aplikasi setelah update malah j...
5,4,1101,4_wajah_verifikasi wajah_verifikasi_wajah gagal,"[wajah, verifikasi wajah, verifikasi, wajah ga...","[verifikasi wajah selalu gagal terus, enggak b..."
6,5,971,5_gangguan_sering_enggak jelas_mulu,"[gangguan, sering, enggak jelas, mulu, jelas, ...","[parah sering banget gangguan enggak jelas, ap..."
7,6,897,6_daftar_susah_mau daftar_mau,"[daftar, susah, mau daftar, mau, susah banget,...","[mau daftar saja susah my meminta ampun, mau d..."
8,7,734,7_otomatis_pengaturan_jam_pengaturan jam,"[otomatis, pengaturan, jam, pengaturan jam, ja...",[kenapa pada saat melakukan transaksi selalu m...
9,8,671,8_otp_kode otp_kode_otp enggak,"[otp, kode otp, kode, otp enggak, otp nya, kir...","[kenapa kode otp nya enggak sampai sampai, ken..."


In [130]:
num_topics = len(topic_info) - 1

outlier_count = (np.array(topics) == -1).sum()

outlier_percentage = (
    outlier_count / len(topics)
) * 100

print(f"Topics              : {num_topics}")
print(f"Outliers            : {outlier_count:,}")
print(f"Outlier Percentage  : {outlier_percentage:.2f}%")

Topics              : 70
Outliers            : 142,860
Outlier Percentage  : 84.71%


Topic Size

In [131]:
topic_info[["Topic","Count"]]

,Topic,Count
0,-1,142860
1,0,1902
2,1,1800
3,2,1564
4,3,1526
...,...,...
66,65,106
67,66,106
68,67,104
69,68,104


Top Words

In [132]:
for topic in topic_info.Topic:

    if topic == -1:
        continue

    print("="*80)

    print(f"Topic {topic}")

    print(topic_model.get_topic(topic))

Topic 0
[('username', np.float64(0.0610858829939712)), ('benar', np.float64(0.05992589923036017)), ('password', np.float64(0.054207903375843275)), ('salah', np.float64(0.048931714179079246)), ('pasword', np.float64(0.04185146435015865)), ('sandi', np.float64(0.0318547010034882)), ('login', np.float64(0.02954066181624653)), ('padahal', np.float64(0.028514361359703283)), ('padahal username', np.float64(0.028421842922908353)), ('username password', np.float64(0.028095709748775698))]
Topic 1
[('buka', np.float64(0.04628244365247576)), ('dibuka', np.float64(0.04460260003986699)), ('enggak buka', np.float64(0.040181482298078615)), ('kok', np.float64(0.03393815211563264)), ('enggak dibuka', np.float64(0.030821852497289644)), ('aplikasi', np.float64(0.027985485879477287)), ('aplikasinya', np.float64(0.02749814007248107)), ('kok enggak', np.float64(0.025095522247014904)), ('download', np.float64(0.023987010947475813)), ('enggak', np.float64(0.023986184533594475))]
Topic 2
[('perbaiki', np.float

Representative Reviews

In [133]:
representative_docs = topic_model.get_representative_docs()

for topic in representative_docs:

    if topic == -1:
        continue

    print("="*100)

    print(f"Topic {topic}")

    print()

    for doc in representative_docs[topic][:5]:

        print("-", doc)

    print()

Topic 0

- kenapa enggak bisa login padahal username sama password sudah benar
- kenapa ya enggak bisa login padahal username dan password sudah benar tapi tetap tidak bisa
- update malah enggak bisa login padahal username dan password benar

Topic 1

- kenapa enggak bisa di buka saat ini aplikasi nya
- kenapa kok enggak bisa buka aplikasi ini
- kok enggak bisa di buka aplikasi nya

Topic 2

- bug force close terus ni mohon segera diperbaiki lagi
- maaf saya kasih bintang 3 dulu karena masih sering keluar sendiri mohon segera di perbaiki
- kenapa setelah di update aplikasi sering keluar sendiri mohon segera diperbaiki

Topic 3

- bagaimana ini aplikasi setelah update malah jadi enggak bisa login baru coba login langsung close otomatis terus sudah coba restart hp uninstall dan install ulang pun tetap sama mohon solusinya
- ini kenapa aplikasi nya tiba-tiba enggak bisa dibuka ya saya sudah coba uninstall dan install lagi tetap enggak bisa dibuka
- setelah di update ke versi 4 4 3 selalu 

silhoutte score

In [134]:
from sklearn.metrics import silhouette_score

mask = np.array(topics) != -1

silhouette = silhouette_score(
    topic_model.umap_model.embedding_[mask],
    np.array(topics)[mask]
)

print(f"Silhouette Score : {silhouette:.4f}")

Silhouette Score : 0.4671


In [135]:
from itertools import chain

top_n = 10
topic_words = []

for topic in topic_info.Topic:
    if topic == -1:
        continue

    words = [
        word
        for word, score in topic_model.get_topic(topic)[:top_n]
    ]
    topic_words.append(words)

unique_words = len(
    set(chain.from_iterable(topic_words))
)

total_words = len(topic_words) * top_n
topic_diversity = unique_words / total_words

print(f"Topic Diversity : {topic_diversity:.4f}")

Topic Diversity : 0.6943


Representative Reviews

In [136]:
representative_docs = topic_model.get_representative_docs()

for topic, docs in representative_docs.items():

    if topic == -1:
        continue

    print("="*100)

    print(f"TOPIC {topic}")

    print()

    for i, doc in enumerate(docs[:5],1):

        print(f"{i}. {doc}")

        print()

TOPIC 0

1. kenapa enggak bisa login padahal username sama password sudah benar

2. kenapa ya enggak bisa login padahal username dan password sudah benar tapi tetap tidak bisa

3. update malah enggak bisa login padahal username dan password benar

TOPIC 1

1. kenapa enggak bisa di buka saat ini aplikasi nya

2. kenapa kok enggak bisa buka aplikasi ini

3. kok enggak bisa di buka aplikasi nya

TOPIC 2

1. bug force close terus ni mohon segera diperbaiki lagi

2. maaf saya kasih bintang 3 dulu karena masih sering keluar sendiri mohon segera di perbaiki

3. kenapa setelah di update aplikasi sering keluar sendiri mohon segera diperbaiki

TOPIC 3

1. bagaimana ini aplikasi setelah update malah jadi enggak bisa login baru coba login langsung close otomatis terus sudah coba restart hp uninstall dan install ulang pun tetap sama mohon solusinya

2. ini kenapa aplikasi nya tiba-tiba enggak bisa dibuka ya saya sudah coba uninstall dan install lagi tetap enggak bisa dibuka

3. setelah di update ke

In [146]:
random.seed(42)

sample_size = 20

for topic_id in sorted(set(topics)):

    if topic_id == -1:
        continue

    topic_docs = [
        doc for doc, topic in zip(documents, topics)
        if topic == topic_id
    ]

    n = min(sample_size, len(topic_docs))
    sampled_docs = random.sample(topic_docs, n)

    print("\n" + "=" * 120)
    print(f"TOPIC {topic_id}")
    print(f"CLUSTER SIZE : {len(topic_docs)}")
    print(f"SAMPLE SIZE  : {n}")
    print("=" * 120)

    for i, doc in enumerate(sampled_docs, 1):
        print(f"{i}. {doc}")


TOPIC 0
CLUSTER SIZE : 1902
SAMPLE SIZE  : 20
1. kenapa aplikasi aku enggak bisa di buka ya padahal password sama username sudah benar semua kecewa pakai bri mau
2. setelah di update malah pas mau login gagal terus padahal jelas jelas pakai akun email dan sandi yang biasa dipakai aneh enggak jelas kecewa
3. entah salah saya apa nama pasword sudah benar tapi ketika login enggak bisa2 padahal lagi butuh bantuan brimo lagi
4. aku baru ganti hp malah sekarang enggak bisa bisa login padahal username n password deh benar
5. username sudah benar password juga sudah benar tapi enggak bisa login aplikasi sialan
6. anjng bagaimana sih sudah benar memasuki pasword malah ke blokir
7. kenapa enggak bisa login padahal username sama password benar malah pemberitahuannya data anda tidak aktif dan tidak bisa terkoneksi biasanya itu kendala apa kak
8. ada yang ngalamin kayak aku enggak apk brimo tiba-tiba logout sendiri terus pas memasuki username smaa pasword katanya data enggak sesuai hsan mulu padah

NPMI

In [137]:
analyzer = topic_model.vectorizer_model.build_analyzer()

In [138]:
doc.split()

['semenjak',
 'memakai',
 'apk',
 'livin',
 'ini',
 'sangat',
 'jengkel',
 'mau',
 'masuk',
 'apk',
 'livin',
 'malah',
 'keluar',
 'sendiri',
 'bug',
 'bagaimana',
 'costumer',
 'service',
 'nya',
 'ini',
 'apakah',
 'bisa',
 'diperbaiki',
 'lagi',
 'bug',
 'menyesak',
 'banget',
 'mau',
 'cek',
 'saldo',
 'di',
 'livin',
 'tidak',
 'bisa',
 'keluar',
 'sendiri',
 'saya',
 'kasih',
 'bintang',
 '1',
 'saja',
 'rasa',
 'kecewanya',
 'pun',
 'sangat',
 'berulang',
 'kali',
 'di',
 'update',
 'malah',
 'sama',
 'saja',
 'jengkel',
 'padahal',
 'ada',
 'keperluan',
 'penting',
 'pun',
 'tak',
 'bisa',
 'memakai',
 'apk',
 'livin',
 'ini',
 'mau',
 'transfer',
 'pun',
 'tidak',
 'bisa',
 'karena',
 'tidak',
 'bisa',
 'masuk',
 'ke',
 'apk',
 'livin',
 'nya',
 'tolong',
 'di',
 'tingkatkan',
 'apk',
 'livin',
 'nya']

In [139]:
tokenized_docs = [
    analyzer(doc)
    for doc in documents
]

In [141]:
from gensim.corpora import Dictionary

dictionary = Dictionary(tokenized_docs)
topic_words = []

for topic in topic_info.Topic:

    if topic == -1:
        continue

    words = []

    for word, score in topic_model.get_topic(topic):
        if word in dictionary.token2id:
            words.append(word)
    # Need at least 2 words for coherence
    if len(words) >= 2:
        topic_words.append(words)

In [142]:
# sanity check
print(f"Valid Topics : {len(topic_words)}")

print()

print(topic_words[:3])

Valid Topics : 70

[['username', 'benar', 'password', 'salah', 'pasword', 'sandi', 'login', 'padahal', 'padahal username', 'username password'], ['buka', 'dibuka', 'enggak buka', 'kok', 'enggak dibuka', 'aplikasi', 'aplikasinya', 'kok enggak', 'download', 'enggak'], ['perbaiki', 'diperbaiki', 'mohon', 'segera', 'sering', 'mohon perbaiki', 'close', 'aplikasinya', 'dong', 'update']]


In [143]:
from gensim.models.coherencemodel import CoherenceModel

coherence_model = CoherenceModel(
    topics=topic_words,
    texts=tokenized_docs,
    dictionary=dictionary,
    coherence="c_npmi"
)

npmi = coherence_model.get_coherence()
print(f"NPMI : {npmi:.4f}")

NPMI : 0.1462


DBCV

In [145]:
mask = np.array(topics) != -1
X = topic_model.umap_model.embedding_[mask].astype(np.float64)
labels = np.array(topics)[mask]

dbcv_score = validity_index(X, labels)
print(f"DBCV : {dbcv_score:.4f}")

DBCV : 0.1246


In [147]:
import pandas as pd
from scipy.stats import chi2_contingency

df["topic"] = topics

# 1. Baseline: proporsi tiap bank di keseluruhan korpus
baseline = df["bank"].value_counts(normalize=True) * 100
print("Proporsi bank di keseluruhan korpus (baseline):")
print(baseline.round(2))
print()

# 2. Proporsi tiap bank DI DALAM tiap topik
crosstab = pd.crosstab(df["topic"], df["bank"], normalize="index") * 100
crosstab = crosstab.round(2)

# 3. Hitung "lift" = proporsi di topik / proporsi baseline
#    >1 artinya over-represented di topik itu, <1 artinya under-represented
lift = crosstab.copy()
for bank in baseline.index:
    lift[bank] = crosstab[bank] / baseline[bank]

# 4. Tandai topik yang "njomplang" (deviasi lift > 1.5x atau < 0.5x dari baseline)
def flag_imbalance(row):
    return any(row > 1.5) or any(row < 0.5)

lift["is_imbalanced"] = lift[baseline.index].apply(flag_imbalance, axis=1)

# gabung count per topik biar gampang liat mana yang topik "besar" (bukan cuma noise kecil)
topic_sizes = df[df["topic"] != -1]["topic"].value_counts()
lift["topic_size"] = lift.index.map(topic_sizes)

result = lift[lift.index != -1].sort_values("is_imbalanced", ascending=False)
print(result[list(baseline.index) + ["is_imbalanced", "topic_size"]])

Proporsi bank di keseluruhan korpus (baseline):
bank
BRIMO_REVIEWS            36.98
LIVIN_MANDIRI_REVIEWS    32.33
BCAMOBILE_REVIEWS        19.99
WONDR_BNI_REVIEWS        10.69
Name: proportion, dtype: float64

bank   BRIMO_REVIEWS  LIVIN_MANDIRI_REVIEWS  BCAMOBILE_REVIEWS  \
topic                                                            
0           2.351386               0.263511           0.073525   
49          0.499967               1.334564           1.267436   
32          0.562970               2.284996           0.183563   
33          0.570541               1.291883           0.991841   
34          0.981006               0.780016           1.837132   
...              ...                    ...                ...   
39          0.882311               1.253531           0.789772   
23          1.042927               1.076620           0.904811   
36          0.880688               0.978886           1.445497   
1           1.025892               1.116827           0.928319 